# Lab 09-03: Detect and Break a Reasoning Loop

**Book chapter:** Chapter 09: Deploying, Operating, and Observing Agents  
**Estimated time:** 15 minutes

## Overview

Cymbal Retail's operations team notices refund conversations with many tool calls and
high token use. In this lab, you reproduce a reasoning loop, stop it with a limit on
model calls, detect it with a before-tool callback, and fix its root cause in the tool
response and the instruction.

## Learning Objectives

By the end of this lab, you will be able to:

- Reproduce a reasoning loop and measure its cost
- Cap model calls per invocation with RunConfig
- Detect repeated identical tool calls in a callback
- Fix a loop with actionable tool responses and completion criteria

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. No resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Defense | Type | What it does |
|---|---|---|
| `max_llm_calls` | Safety net | Stops any invocation after a fixed number of model calls |
| Loop detector callback | Safety net | Blocks repeated identical tool calls with a clear error |
| Clear tool responses and instructions | Root-cause fix | Tells the agent when the task is complete |

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Define a tool, a looping agent, and a test harness

`check_refund_status` always returns `processing`, and the `LOOPING` instruction tells
the agent to keep checking until the refund is complete, which can never happen. The `run`
helper limits each run with `RunConfig(max_llm_calls=10)` and counts tool calls and
tokens.

In [ ]:
import collections
import logging
import os
import uuid

from google.adk.agents import Agent
from google.adk.agents.invocation_context import LlmCallsLimitExceededError
from google.adk.agents.run_config import RunConfig
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
QUESTION = "Has the refund for order 10421 been completed?"
RESULTS = []
# ADK logs a run stopped by the limit as an error with a traceback. The
# harness reports the stop itself, and errors are still raised, so hide
# ADK's error log entries.
logging.getLogger("google_adk").setLevel(logging.CRITICAL)


def check_refund_status(order_id: str) -> dict:
    """Checks the status of a refund."""
    return {"order_id": order_id, "status": "processing"}


LOOPING = ("You help shoppers with refunds. Keep calling "
           "check_refund_status until the refund is complete. Never "
           "answer before the refund is complete.")


def run(agent, max_llm_calls=10):
    """Runs one question with a cap on model calls and reports the cost."""
    runner = Runner(app_name="refunds", agent=agent,
                    session_service=InMemorySessionService(),
                    auto_create_session=True)
    message = types.Content(role="user", parts=[types.Part(text=QUESTION)])
    calls, tokens, answer = 0, 0, ""
    try:
        for event in runner.run(
                user_id="lab-user", session_id=str(uuid.uuid4()),
                new_message=message,
                run_config=RunConfig(max_llm_calls=max_llm_calls)):
            calls += len(event.get_function_calls())
            if event.usage_metadata:
                tokens += event.usage_metadata.total_token_count or 0
            if event.is_final_response() and event.content:
                answer = event.content.parts[0].text or ""
    except LlmCallsLimitExceededError:
        answer = f"STOPPED after {max_llm_calls} model calls"
    RESULTS.append((agent.name, calls, tokens))
    print(f"{agent.name}: {calls} tool calls, {tokens} tokens\n"
          f"   {answer.strip()}\n")

## Step 2: Run the looping agent with a limit on model calls

Run the looping agent.

In [ ]:
looping = Agent(name="looping_agent", model=MODEL, instruction=LOOPING,
                tools=[check_refund_status])
run(looping)

**Expected result:** many identical tool calls, usually until the run stops at 10 model calls.

## Step 3: Detect repeated identical tool calls in a callback

A **before-tool callback** sees every tool call before it runs. `stop_repeats` counts
identical calls in the same invocation and, on the third, returns an actionable error
instead of running the tool.

In [ ]:
REPEATS = collections.Counter()


def stop_repeats(tool, args, tool_context):
    """Blocks the third identical call in the same invocation."""
    key = (tool_context.invocation_id, tool.name, str(sorted(args.items())))
    REPEATS[key] += 1
    if REPEATS[key] >= 3:
        print(f"   [loop detector] blocked call {REPEATS[key]} to "
              f"{tool.name}")
        return {"error": "Repeated call blocked: the status will not "
                         "change during this conversation. Tell the "
                         "shopper the current status and stop."}
    return None  # Let the tool run.


guarded = Agent(name="guarded_agent", model=MODEL, instruction=LOOPING,
                tools=[check_refund_status],
                before_tool_callback=stop_repeats)
run(guarded)

**Expected result:** the detector blocks call 3, and the agent reports the current status.

## Step 4: Fix the root cause in the tool and the instruction

The real fix is in the design: the tool says when the status will change, and the
instruction says that a processing refund is a complete answer.

In [ ]:
def check_refund_status(order_id: str) -> dict:
    """Checks the status of a refund. The status changes once a day."""
    return {"order_id": order_id, "status": "processing",
            "expected_by": "Friday",
            "note": "Final status for today. Do not check again."}


FIXED = ("You help shoppers with refunds. Call check_refund_status once, "
         "and then tell the shopper the status and when to expect the "
         "refund. A refund that is still processing is a complete answer.")
fixed = Agent(name="fixed_agent", model=MODEL, instruction=FIXED,
              tools=[check_refund_status],
              before_tool_callback=stop_repeats)  # Keep the safety net.
run(fixed)

**Expected result:** one tool call and an answer that gives the expected date.

## Step 5: Compare the three versions

Compare tool calls and tokens.

In [ ]:
print(f"{'Agent':<15}{'Tool calls':>11}{'Tokens':>9}")
for name, calls, tokens in RESULTS:
    print(f"{name:<15}{calls:>11}{tokens:>9}")

**What to look for:** tool calls and tokens fall from the looping agent to the fixed agent. High tool calls per invocation is the metric that reveals loops in production.

## Step 6: Experiment

- Rerun Step 2 with `run(looping, max_llm_calls=20)`, and notice that the cost of an
  unbroken loop grows with the limit. ADK's default limit is 500 model calls.
- In Step 4, remove `before_tool_callback` from the fixed agent, rerun the step, and confirm
  that the fix alone keeps it to one tool call.

## Check Your Understanding

**1. Which production signals point to a reasoning loop?**

<details><summary>Show answer</summary>

Repeated identical tool spans in traces, and high tool calls or tokens per invocation in metrics.

</details>

**2. Why keep the loop detector after fixing the root cause?**

<details><summary>Show answer</summary>

It is a safety net. A future change to a tool, model, or instruction could cause a new loop, and the detector limits its cost.

</details>

**3. What makes a tool error actionable?**

<details><summary>Show answer</summary>

It tells the model what happened and what to do next, so the model doesn't retry blindly.

</details>

## Summary

You reproduced a reasoning loop, contained it with a call limit and a loop detector, and fixed
its cause. This lab created no resources, so there is nothing to clean up.